In [19]:
import _referAsMain

import math
import time
import random
import sys
import numpy
from IPython.display import SVG, display
from holo.pointers import Pointer
from holo.profilers import Profiler
from holo.prettyFormats import prettyPrint, prettyTime
print(sys.version_info)

import torch
from torch.utils.data import DataLoader

from paths_cfg import TOKENIZER_SAVE_DIRECTORY, OUR_DATASET_DIRECTORY, OUR_DATASET_DIRECTORY_2, OUR_DATASET_DIRECTORY_3
from dataset import svg_dataset
import tokenizer_pfe.tokenizer_project as tokenizerLib
import metrics.metrics

from LLM.model import Model, GenerationStats
import LLM.nanochat.gpt as nanoChatModel
from LLM.nanochat.common import compute_init, autodetect_device_type

sys.version_info(major=3, minor=13, micro=12, releaselevel='final', serial=0)


In [2]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Torch version: 2.9.1+cu128
CUDA available: True
CUDA version: 12.8
Device count: 1


In [3]:
import importlib
import LLM.model
_ = importlib.reload(LLM.model)
from LLM.model import Model

In [9]:
try:
    torch.cuda.empty_cache()
    del model # type: ignore
    torch.cuda.empty_cache()
except Exception: pass
model = Model.load("gcode_100", versionID=3, device=torch.device("cuda:0"))
model.show_infos()

loading the tokenizer from: /autofs/unitytravail/travail/lcurtis/M2INFO/PFE/PFE_LLM_art_generation/models_save/gcode_100/tokenizer.json
loading the historique from: /autofs/unitytravail/travail/lcurtis/M2INFO/PFE/PFE_LLM_art_generation/models_save/gcode_100/versions/version_0003_checkpoint-1/history.json
Padding vocab_size from 585 to 640 for efficiency
Scaling the LR for the AdamW parameters ∝1/√(384/768) = 1.414214
GPTConfig(sequence_len=8192, vocab_size=585, n_layer=6, n_head=3, n_kv_head=3, n_embd=384, window_pattern='SSSL')
11_845_932 total params (embeding: 983_040 | last layer: 245_760 | transformer: 10_617_120)
on device: device(type='cuda', index=0), with effective context_size: 4096


In [10]:
print(f"trained for {model.nb_epoches_done} epochs")


trained for 1 epochs


In [11]:
try:
    if "dataset" in globals() : raise FileExistsError
    # TO DO : ajouter les param du gcode ici
    dataset = svg_dataset.SVGDataset(
        OUR_DATASET_DIRECTORY, context_size=model.context_size,
        tokenizer=model.tokenizer.encode, decoder=model.tokenizer.decode, use_gcode = True, use_relative_gcode=False)
except FileExistsError: pass

In [12]:
N = 13
print(f"using: {dataset.samples[N].svg_file}")
#start = dataset.samples[N].txt[: model.context_size]
#print(start); break
first_chunck = next(c for c in dataset.chunks if c.indexes.svgIndex==N)
start = first_chunck.text
statsPtr: Pointer[GenerationStats] = Pointer()
results = []
for txt in model.generate_flow(
        start=start, decode_batch=256, temperature=1.0, top_k=5, 
        max_tokens=320000, max_time=2*60, statsPtr=statsPtr):
    results.append(txt)
    print(txt, end="", flush=True)

using: /autofs/unitytravail/travail/lcurtis/M2INFO/PFE/PFE_LLM_art_generation/dataset/samples_100/0013_circle_packing.svg


W0316 15:51:40.042000 874417 torch/_inductor/utils.py:1613] [0/0] Not enough SMs to use max_autotune_gemm mode


658""/><line y2="0.863" style="fill:none;" x1="354.271" x2="363.253" y1="5.915"/><line y2="0.868" style="fill:none;" x1="358.285" x2="386.278" y1="0.865"/><line y2="0.855" style="fill:none;" x1="363.258" x2="383.084" y1="0.865"/><line y2="0.865" style="fill:none;" x1="363.258" x2="392.0751" y1="0.865"/><line y2="0.865" style="fill:none;" x1="390.278" x2="398.265" y1="0.865"/><line y2="0.873" style="fill:none;" x1="398.265" x2="408.2639" y1="0.865"/><line y2="0.865" style="fill:none;" x1="390.865" x2="397.0723" y1="409.258"/><line y2="405.6584" style="fill:none;" x1="397.0723" x2="396.9384" y1="400.865"/><line y2="398.261" style="fill:none;" x1="408.2639" x2="415.6111" y1="408.2834"/><line y2="396.0811" style="fill:none;" x1="415.6111" x2="415.6066" y1="397.084"/><line y2="392.2899" style="fill:none;" x1="415.6111" x2="408.0811" y1="395.0811"/><line y2="393.0309" style="fill:none;" x1="403.6209" x2="401.0807" y1="408.6223"/><line y2="391.0707" style="fill:none;" x1="400.0811" x2="408.28

In [13]:
print(start)

<|output_start|>G20
G17
G90
G00 X0.0000 Y9.3750
G01 X9.3750 Y9.3750
G01 X9.3750 Y0.0000
G01 X0.0000 Y0.0000
G01 X0.0000 Y9.3750
G00 X5.9714 Y8.0723
G01 X5.9714 Y8.0683
G01 X5.9713 Y8.0644
G01 X5.9711 Y8.0604
G01 X5.9708 Y8.0565
G01 X5.9704 Y8.0526
G01 X5.9700 Y8.0486
G01 X5.9695 Y8.0447
G01 X5.9689 Y8.0408
G01 X5.9683 Y8.0369
G01 X5.9675 Y8.0330
G01 X5.9667 Y8.0292
G01 X5.9659 Y8.0253
G01 X5.9649 Y8.0215
G01 X5.9639 Y8.0177
G01 X5.9628 Y8.0139
G01 X5.9616 Y8.0102
G01 X5.9603 Y8.0064
G01 X5.9590 Y8.0027
G01 X5.9576 Y7.9990
G01 X5.9561 Y7.9954
G01 X5.9546 Y7.9917
G01 X5.9529 Y7.9881
G01 X5.9513 Y7.9846
G01 X5.9495 Y7.9810
G01 X5.9477 Y7.9775
G01 X5.9458 Y7.9741
G01 X5.9438 Y7.9706
G01 X5.9418 Y7.9673
G01 X5.9397 Y7.9639
G01 X5.9375 Y7.9606
G01 X5.9353 Y7.9573
G01 X5.9330 Y7.9541
G01 X5.9307 Y7.9510
G01 X5.9283 Y7.9478
G01 X5.9258 Y7.9448
G01 X5.9233 Y7.9417
G01 X5.9207 Y7.9388
G01 X5.9180 Y7.9358
G01 X5.9153 Y7.9330
G01 X5.9125 Y7.9302
G01 X5.9097 Y7.9274
G01 X5.9069 Y7.9247
G01 X5.9039 

In [14]:
results.insert(0, start)
print(statsPtr.value)
print(f" -> {statsPtr.value.nb_tokens / statsPtr.value.gen_time:.2f} tokens/sec")

GenerationStats(nb_tokens=10094, gen_time=120.00678023399996, stop_reason='reached max_time')
 -> 84.11 tokens/sec


In [15]:
print(results)


['<|output_start|>G20\nG17\nG90\nG00 X0.0000 Y9.3750\nG01 X9.3750 Y9.3750\nG01 X9.3750 Y0.0000\nG01 X0.0000 Y0.0000\nG01 X0.0000 Y9.3750\nG00 X5.9714 Y8.0723\nG01 X5.9714 Y8.0683\nG01 X5.9713 Y8.0644\nG01 X5.9711 Y8.0604\nG01 X5.9708 Y8.0565\nG01 X5.9704 Y8.0526\nG01 X5.9700 Y8.0486\nG01 X5.9695 Y8.0447\nG01 X5.9689 Y8.0408\nG01 X5.9683 Y8.0369\nG01 X5.9675 Y8.0330\nG01 X5.9667 Y8.0292\nG01 X5.9659 Y8.0253\nG01 X5.9649 Y8.0215\nG01 X5.9639 Y8.0177\nG01 X5.9628 Y8.0139\nG01 X5.9616 Y8.0102\nG01 X5.9603 Y8.0064\nG01 X5.9590 Y8.0027\nG01 X5.9576 Y7.9990\nG01 X5.9561 Y7.9954\nG01 X5.9546 Y7.9917\nG01 X5.9529 Y7.9881\nG01 X5.9513 Y7.9846\nG01 X5.9495 Y7.9810\nG01 X5.9477 Y7.9775\nG01 X5.9458 Y7.9741\nG01 X5.9438 Y7.9706\nG01 X5.9418 Y7.9673\nG01 X5.9397 Y7.9639\nG01 X5.9375 Y7.9606\nG01 X5.9353 Y7.9573\nG01 X5.9330 Y7.9541\nG01 X5.9307 Y7.9510\nG01 X5.9283 Y7.9478\nG01 X5.9258 Y7.9448\nG01 X5.9233 Y7.9417\nG01 X5.9207 Y7.9388\nG01 X5.9180 Y7.9358\nG01 X5.9153 Y7.9330\nG01 X5.9125 Y7.9302\nG

In [ ]:
import importlib
from dataset import svg_dataset

importlib.reload(svg_dataset) # pour mettre le fichier py a jour dazns notebook

clean = svg_dataset.clean_gcode(results)
print(clean)

G20
G17
G90
G00 X0.0000 Y9.3750
G01 X9.3750 Y9.3750
G01 X9.3750 Y0.0000
G01 X0.0000 Y0.0000
G01 X0.0000 Y9.3750
G00 X5.9714 Y8.0723
G01 X5.9714 Y8.0683
G01 X5.9713 Y8.0644
G01 X5.9711 Y8.0604
G01 X5.9708 Y8.0565
G01 X5.9704 Y8.0526
G01 X5.9700 Y8.0486
G01 X5.9695 Y8.0447
G01 X5.9689 Y8.0408
G01 X5.9683 Y8.0369
G01 X5.9675 Y8.0330
G01 X5.9667 Y8.0292
G01 X5.9659 Y8.0253
G01 X5.9649 Y8.0215
G01 X5.9639 Y8.0177
G01 X5.9628 Y8.0139
G01 X5.9616 Y8.0102
G01 X5.9603 Y8.0064
G01 X5.9590 Y8.0027
G01 X5.9576 Y7.9990
G01 X5.9561 Y7.9954
G01 X5.9546 Y7.9917
G01 X5.9529 Y7.9881
G01 X5.9513 Y7.9846
G01 X5.9495 Y7.9810
G01 X5.9477 Y7.9775
G01 X5.9458 Y7.9741
G01 X5.9438 Y7.9706
G01 X5.9418 Y7.9673
G01 X5.9397 Y7.9639
G01 X5.9375 Y7.9606
G01 X5.9353 Y7.9573
G01 X5.9330 Y7.9541
G01 X5.9307 Y7.9510
G01 X5.9283 Y7.9478
G01 X5.9258 Y7.9448
G01 X5.9233 Y7.9417
G01 X5.9207 Y7.9388
G01 X5.9180 Y7.9358
G01 X5.9153 Y7.9330
G01 X5.9125 Y7.9302
G01 X5.9097 Y7.9274
G01 X5.9069 Y7.9247
G01 X5.9039 Y7.9220
G01 X5.9